<a href="https://colab.research.google.com/github/lnambiar18/Nambiar_Lohit_HW3_DSCI552/blob/main/Nambiar_Lohit_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1>Nambiar_Lohit_HW3</h1></center>
<br>
<br>

Name: Lohit Nambiar
<br>
Github Username: lnambiar18
<br>
USC ID: 8395177707

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [6]:
import pandas as pd
import os
import zipfile
import numpy as np

Get the AReM Data Set

In [7]:
if not os.path.exists("data/AReM"):
    with zipfile.ZipFile("AReM.zip") as z:
        z.extractall("data")

data_path = os.path.join("data", "AReM")

### (b) Test and Train Data

In [8]:
folders = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

train_data = []
test_data = []

for folder in folders:
    folder_path = os.path.join(data_path, folder)
    files = os.listdir(folder_path)

    for file in files:
        if file.endswith(".csv"):

            file_path = os.path.join(folder_path, file)

            df = pd.read_csv(file_path, skiprows=4, engine="python", on_bad_lines="skip")
            df = df.iloc[:, 1:7]

            df = df.apply(pd.to_numeric, errors="coerce")

            dataset_number = int(file.replace("dataset", "").replace(".csv", ""))

            df["activity"] = folder
            df["dataset"] = dataset_number

            if folder == "bending1" or folder == "bending2":
                test_numbers = [1, 2]
            else:
                test_numbers = [1, 2, 3]

            if dataset_number in test_numbers:
                test_data.append(df)
            else:
                train_data.append(df)

### (c) Feature Extraction

#### i. Research

For time series classification, common time-domain features include the minimum, maximum, mean, median, standard deviation, variance, and range of the values. These features help show the overall behavior of each time series, such as its typical value and how much it changes over time. Other common features include the first and third quartiles, skewness, kurtosis, and the number of peaks or zero crossings.

#### ii. Extraction

In [9]:
sensor_columns = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

all_data = train_data + test_data

feature_data = []

for df in all_data:
    row = {}
    row["activity"] = df["activity"].iloc[0]
    row["dataset"] = df["dataset"].iloc[0]

    for column in sensor_columns:
        row["min_" + column] = df[column].min()
        row["max_" + column] = df[column].max()
        row["mean_" + column] = df[column].mean()
        row["median_" + column] = df[column].median()
        row["std_" + column] = df[column].std()
        row["first_quartile_" + column] = df[column].quantile(0.25)
        row["third_quartile_" + column] = df[column].quantile(0.75)

    feature_data.append(row)

feature_df = pd.DataFrame(feature_data)

feature_df.head()

,activity,dataset,min_avg_rss12,max_avg_rss12,mean_avg_rss12,median_avg_rss12,std_avg_rss12,first_quartile_avg_rss12,third_quartile_avg_rss12,min_var_rss12,...,std_avg_rss23,first_quartile_avg_rss23,third_quartile_avg_rss23,min_var_rss23,max_var_rss23,mean_var_rss23,median_var_rss23,std_var_rss23,first_quartile_var_rss23,third_quartile_var_rss23
0,bending1,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
1,bending1,7,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83
2,bending1,6,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
3,bending1,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
4,bending1,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94


#### iii. Standard Deviation

In [10]:
np.random.seed(1018)

features = feature_df.columns[2:]
results = []

for feature in features:

    values = feature_df[feature].dropna()
    original_std = np.std(values, ddof=1)

    bootstrap_stds = []

    for i in range(1000):
        sample = np.random.choice(values, size=len(values), replace=True)
        sample_std = np.std(sample, ddof=1)
        bootstrap_stds.append(sample_std)

    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)

    results.append([feature, original_std, lower, upper])

confidence_int_df = pd.DataFrame(results, columns=["Feature", "Standard Deviation", "Lower Bound", "Upper Bound"])

confidence_int_df

,Feature,Standard Deviation,Lower Bound,Upper Bound
0,min_avg_rss12,9.624011,8.376366,10.866073
1,max_avg_rss12,4.207745,3.117920,5.058236
2,mean_avg_rss12,5.276413,4.670961,5.851407
3,median_avg_rss12,5.386624,4.756462,5.937458
4,std_avg_rss12,1.771313,1.561738,1.951062
5,first_quartile_avg_rss12,6.128143,5.546590,6.638081
6,third_quartile_avg_rss12,5.031028,4.161363,5.713213
7,min_var_rss12,0.000000,0.000000,0.000000
8,max_var_rss12,5.059656,4.610026,5.401704
9,mean_var_rss12,1.577941,1.416555,1.700047


#### iv. Select Features

The features I am selecting are the minimum, mean, and maximum as the three most important features because together they show the lowest, typical, and highest sensor values for each activity.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression will have a lower or equal training RSS than the linear regression. This is because the cubic model is more flexible and can still in theory act like a linear model by making its X^2 and X^3 coeeficents near 0 to the point where the function just acts like a linear regression function.

### (b) Linear Test

For test RSS, I would expect the linear regression to be lower. Since the true relationship is linear, the flexibility in the cubic model would most likely just fit to random noise.

### (c) Not Linear Train

The cubic regression will have a lower or equal training RSS. Even if the true relationship is not linear, the cubic model has more flexibility, so it can fit the training data as well as the linear model.

### (d) Not Linear Testing

There is not enough information to tell which model would have the lower test RSS. It depends on how non-linear the actual relationship is, since a cubic model could help if the relationship is very curved but could also overfit if the data relationship is actually a very little bit non-linear.

## 3. ISLR 3.7.3 - Extra Practice

## 3. ISLR 3.7.5 - Extra Practice